# 1. Import and Hardware Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split, Subset
import os
import random
import matplotlib.pyplot as plt
import numpy as np
!pip install tqdm -q
from tqdm.auto import tqdm
# Set device to GPU, MPS, or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
DATA_PATH = './data'

# 2. Hyperparameter

In [ ]:
BATCH_SIZE = 256
IMG_SIZE = 128
IN_CHANNELS = 3

LR = 1e-4
EPOCHS = 100
SEED = 42
NUM_CLASSES = 50

LATENT_DIM = 512
ENCODER_CHANNELS = [32, 64, 128, 256, 512]

# Dimensionality of the class label embedding vector.
# This embedding is concatenated with the input or latent vector
# to condition the encoder and decoder on the class label.
LABEL_EMBED_DIM = 128

# Weight for the KL divergence term in the VAE loss.
# A higher value encourages a more structured latent space
# but may reduce reconstruction quality.
KL_WEIGHT = 1e-4


# 3. Data Preparation

In [ ]:
def set_seed(seed: int = 42):
    """Set all random seeds for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

def seed_worker(worker_id):
    """Seed function for DataLoader workers to ensure reproducibility."""
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE + 32),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE + 32),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
    ]
)

# Apply seed
set_seed(SEED)
train_generator = torch.Generator().manual_seed(SEED)
eval_generator = torch.Generator().manual_seed(SEED)

dummy_data = datasets.Food101(root=DATA_PATH, split="train", download=True)

filtered_indices = [i for i, label in enumerate(dummy_data._labels) if label < NUM_CLASSES]

train_size = int(0.8 * len(filtered_indices))
val_size = len(filtered_indices) - train_size
split_generator = torch.Generator().manual_seed(SEED)

filtered_dummy_data = Subset(dummy_data, filtered_indices)
train_tmp_subset, val_tmp_subset = random_split(
    filtered_dummy_data, [train_size, val_size], generator=split_generator
)

train_indices = [filtered_indices[i] for i in train_tmp_subset.indices]
val_indices = [filtered_indices[i] for i in val_tmp_subset.indices]

train_dataset = datasets.Food101(
    root=DATA_PATH,
    split="train",
    download=False,
    transform=train_transform,
)

val_dataset = datasets.Food101(
    root=DATA_PATH,
    split="train",
    download=False,
    transform=val_transform,
)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)


dummy_test_dataset = datasets.Food101(root=DATA_PATH, split="test", download=True)
test_filterd_indices = [
    i for i, label in enumerate(dummy_test_dataset._labels) if label < NUM_CLASSES
]

test_dataset_full = datasets.Food101(
    root=DATA_PATH,
    split="test",
    download=False,
    transform=val_transform,
)
test_dataset = Subset(test_dataset_full, test_filterd_indices)


In [ ]:
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True,
    prefetch_factor=10,
    worker_init_fn=seed_worker,
    generator=train_generator,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True,
    prefetch_factor=10,
    worker_init_fn=seed_worker,
    generator=eval_generator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True,
    prefetch_factor=10,
    worker_init_fn=seed_worker,
    generator=eval_generator,
)


# 4. Model Architecture

## CVAE Key Differences vs. Standard VAE

1. **Conditional input**: Both encoder and decoder receive the class label as an additional input, allowing the model to learn class-specific latent distributions.
2. **Label embedding**: Class labels are mapped to a dense embedding vector (LABEL_EMBED_DIM) which is then projected to the appropriate spatial dimensions and concatenated channel-wise with the image (encoder) or with the latent vector (decoder).
3. **Encoder input**: The encoder receives a tensor of shape (B, IN_CHANNELS + 1, H, W), where the extra channel is the spatially broadcast class embedding.
4. **Decoder input**: The decoder receives z concatenated with the label embedding, i.e. a vector of size LATENT_DIM + LABEL_EMBED_DIM.
5. **Generative capability**: After training, we can sample z ~ N(0, I) and provide a specific class label to generate images of that class.


In [ ]:
class ConvBA(nn.Sequential):
    """Convolutional block with BatchNorm and LeakyReLU activation."""
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=2,  # to reduce the resolution
        padding=1,
    ):
        super().__init__(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )


class ConvTransBA(nn.Sequential):
    """Transposed convolutional block with BatchNorm and LeakyReLU activation."""
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=2,
        padding=1,
        output_padding=1,
    ):
        super().__init__(
            nn.ConvTranspose2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                output_padding=output_padding,
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )


class ConditionalVariationalAutoencoder(nn.Module):
    """
    Conditional Variational Autoencoder (CVAE).

    Unlike a standard VAE, the CVAE conditions both the encoder and decoder
    on a class label. This allows the model to learn class-specific latent
    distributions and enables controlled generation of images for a given class.

    The class label is embedded into a dense vector and:
      - For the encoder: projected to a 1-channel spatial map and concatenated
        channel-wise with the input image.
      - For the decoder: concatenated with the sampled latent vector z before
        decoding.
    """
    def __init__(
        self, in_channels, img_size, encoder_channels, latent_dim,
        num_classes, label_embed_dim,
    ):
        super().__init__()

        self.img_size = img_size
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.label_embed_dim = label_embed_dim

        num_layers = len(encoder_channels)

        # The resolution of the feature map after the encoder
        self.final_h = img_size // (2 ** num_layers)
        self.final_w = img_size // (2 ** num_layers)

        assert self.final_h >= 1 and self.final_w >= 1, "Too many downsamplings"

        # -------------- Label Embedding for Encoder --------------
        # Embeds the class label into a dense vector, then projects it
        # to a 1-channel spatial map (1 x H x W) to be concatenated
        # channel-wise with the input image.
        self.encoder_label_embed = nn.Sequential(
            nn.Embedding(num_classes, label_embed_dim),
            nn.Linear(label_embed_dim, img_size * img_size),
        )

        # -------------- Label Embedding for Decoder --------------
        # Embeds the class label into a dense vector to be concatenated
        # with the latent vector z before decoding.
        self.decoder_label_embed = nn.Sequential(
            nn.Embedding(num_classes, label_embed_dim),
        )

        # -------------- Encoder --------------
        # Input channels: in_channels + 1 (for the label channel)
        encoder_layers = []
        curr_channels = in_channels + 1  # +1 for the label spatial map

        for out_channels in encoder_channels:
            encoder_layers.append(ConvBA(curr_channels, out_channels))
            curr_channels = out_channels

        encoder_layers.append(nn.Flatten())
        self.encoder_conv = nn.Sequential(*encoder_layers)

        # Flattened feature size after all conv layers
        flattened_size = curr_channels * self.final_h * self.final_w

        # CVAE specific: two separate heads for mu and logvar
        self.fc_mu = nn.Linear(flattened_size, latent_dim)
        self.fc_logvar = nn.Linear(flattened_size, latent_dim)

        # -------------- Decoder --------------
        # Input size: latent_dim + label_embed_dim (z concatenated with label embedding)
        decoder_layers = []
        decoder_layers.extend(
            [
                nn.Linear(latent_dim + label_embed_dim, flattened_size),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Unflatten(1, (curr_channels, self.final_h, self.final_w)),
            ]
        )

        rev_channels = list(reversed(encoder_channels))
        for i in range(len(rev_channels) - 1):
            curr_channels = rev_channels[i]
            out_channels = rev_channels[i + 1]
            decoder_layers.append(ConvTransBA(curr_channels, out_channels))

        decoder_layers.extend(
            [
                nn.ConvTranspose2d(
                    in_channels=rev_channels[-1],
                    out_channels=in_channels,
                    kernel_size=3,
                    stride=2,
                    padding=1,
                    output_padding=1,
                ),
                nn.Sigmoid(),
            ]
        )

        self.decoder = nn.Sequential(*decoder_layers)

    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick:
        Sample z from the latent distribution N(mu, sigma^2)
        in a way that allows gradient backpropagation.
        z = mu + std * epsilon, where epsilon ~ N(0, 1)
        """
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        return mu + std * epsilon

    def encode(self, x, labels):
        """
        Encode input image conditioned on class label.

        The label is embedded into a 1-channel spatial map and concatenated
        channel-wise with the input image before passing through the encoder.

        Returns:
            mu: Mean of the latent distribution.
            logvar: Log-variance of the latent distribution.
        """
        # Create label spatial map: (B, 1, H, W)
        label_map = self.encoder_label_embed(labels)  # (B, H*W)
        label_map = label_map.view(-1, 1, self.img_size, self.img_size)

        # Concatenate image and label map along channel dimension
        x = torch.cat([x, label_map], dim=1)  # (B, C+1, H, W)

        h = self.encoder_conv(x)
        mu = self.fc_mu(h)
        # Clamp logvar to prevent exp() overflow in mixed precision (float16).
        # float16 max is ~65504, so exp(x) overflows for x > ~11.
        # Without clamping, large logvar values cause exp(logvar) = inf -> NaN.
        logvar = torch.clamp(self.fc_logvar(h), min=-20.0, max=20.0)
        return mu, logvar

    def decode(self, z, labels):
        """
        Decode a latent vector z conditioned on class label.

        The label embedding is concatenated with z before decoding.

        Returns:
            Reconstructed image.
        """
        # Get label embedding: (B, label_embed_dim)
        label_emb = self.decoder_label_embed(labels)

        # Concatenate z and label embedding
        z_cond = torch.cat([z, label_emb], dim=1)  # (B, latent_dim + label_embed_dim)

        return self.decoder(z_cond)

    def forward(self, x, labels):
        """
        Forward pass: encode (conditioned) -> reparameterize -> decode (conditioned).
        Returns: (reconstructed, mu, logvar)
        """
        mu, logvar = self.encode(x, labels)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z, labels)
        return reconstructed, mu, logvar


In [ ]:
model = ConditionalVariationalAutoencoder(
    in_channels=IN_CHANNELS,
    img_size=IMG_SIZE,
    encoder_channels=ENCODER_CHANNELS,
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
    label_embed_dim=LABEL_EMBED_DIM,
).to(device)

print(f"Total parameters: {(sum(p.numel() for p in model.parameters()) / 1e6):.2f}M")


# 5. Loss Function

In [ ]:
def cvae_loss(reconstructed, original, mu, logvar, kl_weight=1e-4):
    """
    CVAE loss = Reconstruction Loss + KL Divergence.

    The loss is identical to the standard VAE loss. The conditioning
    on class labels is handled by the model architecture, not the loss.

    - Reconstruction Loss: MSE between original and reconstructed images.
    - KL Divergence: Measures how much the learned latent distribution
      deviates from a standard normal distribution N(0, I).
      KL(q(z|x,c) || p(z)) = -0.5 * mean(1 + logvar - mu^2 - exp(logvar))

    Args:
        reconstructed: Reconstructed images from the decoder.
        original: Original input images.
        mu: Mean of the latent distribution.
        logvar: Log-variance of the latent distribution.
        kl_weight: Scaling factor for the KL divergence term.

    Returns:
        total_loss: Weighted sum of reconstruction loss and KL divergence.
        recon_loss: Reconstruction loss (MSE) for logging.
        kl_loss: KL divergence for logging.
    """
    # Reconstruction loss (MSE, per-pixel average)
    # Using reduction='mean' for numerical stability under mixed precision
    # (float16). Large sums from reduction='sum' easily overflow float16.
    recon_loss = F.mse_loss(reconstructed, original, reduction='mean')

    # KL Divergence: -0.5 * mean(1 + logvar - mu^2 - exp(logvar))
    # Using mean instead of sum to keep values in a stable range.
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + kl_weight * kl_loss
    return total_loss, recon_loss, kl_loss


# 6. Train

In [ ]:
class EarlyStopping:
    """Early stopping to halt training when validation loss stops improving."""
    def __init__(
        self, patience=10, delta=0, verbose=False, save_path="best_checkpoint.pth"
    ):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.save_path = save_path

        self.early_stop = False
        self.counter = 0
        self.best_loss = None
    
    def __call__(self, model, val_loss):
        # 1. For the first epoch
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        
        # 2. If the loss didnt decrease as expected
        elif val_loss >= self.best_loss - self.delta:
            self.counter += 1
            print(f"Early Stopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        
        # 3. The loss decreased properly
        else:
            self.counter = 0
            self.best_loss = val_loss
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        """Save the model checkpoint."""
        if self.verbose:
            print("Saving best checkpoint ...")
        state_dict = (
            model.module.state_dict()
            if hasattr(model, "module")
            else model.state_dict()
        )
        torch.save(state_dict, self.save_path)


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6,
)
scaler = torch.amp.GradScaler(device=device)


In [ ]:
def train_epoch(model, loader, optimizer, scaler, kl_weight):
    """Train the CVAE for one epoch."""
    model.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    loop = tqdm(loader, desc="Training", leave=False)
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type=device.type):
            reconstructed, mu, logvar = model(images, labels)
            loss, recon_loss, kl_loss = cvae_loss(
                reconstructed, images, mu, logvar, kl_weight
            )
        
        # Scale up the loss and backpropagate
        scaler.scale(loss).backward()
        
        # Unscale and clip the gradients
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update the parameters
        scaler.step(optimizer)
        
        # Update the scaler
        scaler.update()
        
        batch_size = images.size(0)
        total_loss += loss.detach() * batch_size
        total_recon += recon_loss.detach() * batch_size
        total_kl += kl_loss.detach() * batch_size

    n = len(loader.dataset)
    return total_loss.item() / n, total_recon.item() / n, total_kl.item() / n


def validate_epoch(model, loader, kl_weight):
    """Validate the CVAE for one epoch."""
    model.eval()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    loop = tqdm(loader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for images, labels in loop:
            images = images.to(device)
            labels = labels.to(device)
            reconstructed, mu, logvar = model(images, labels)
            loss, recon_loss, kl_loss = cvae_loss(
                reconstructed, images, mu, logvar, kl_weight
            )
            batch_size = images.size(0)
            total_loss += loss.detach() * batch_size
            total_recon += recon_loss.detach() * batch_size
            total_kl += kl_loss.detach() * batch_size

    n = len(loader.dataset)
    return total_loss.item() / n, total_recon.item() / n, total_kl.item() / n


In [ ]:
early_stopping = EarlyStopping(patience=5, save_path="best_cvae_checkpoint.pth")
train_losses = []
val_losses = []
train_recon_losses = []
val_recon_losses = []
train_kl_losses = []
val_kl_losses = []

for epoch in range(EPOCHS):
    train_loss, train_recon, train_kl = train_epoch(
        model, train_loader, optimizer, scaler, KL_WEIGHT
    )
    val_loss, val_recon, val_kl = validate_epoch(
        model, val_loader, KL_WEIGHT
    )
    
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_recon_losses.append(train_recon)
    val_recon_losses.append(val_recon)
    train_kl_losses.append(train_kl)
    val_kl_losses.append(val_kl)
    
    print(
        f"Epoch {epoch+1}/{EPOCHS}: "
        f"Train Loss: {train_loss:.6f} (Recon: {train_recon:.6f}, KL: {train_kl:.2f}) | "
        f"Val Loss: {val_loss:.6f} (Recon: {val_recon:.6f}, KL: {val_kl:.2f})"
    )
    
    early_stopping(model, val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break

# Load best model for evaluation
model.load_state_dict(torch.load("best_cvae_checkpoint.pth"))


# 7. Result Visualization

In [ ]:
# Plot training and validation loss (Total, Reconstruction, and KL)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total Loss
axes[0].plot(train_losses, label="Train Loss")
axes[0].plot(val_losses, label="Val Loss")
axes[0].set_title("CVAE Total Loss")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Reconstruction Loss
axes[1].plot(train_recon_losses, label="Train Recon Loss")
axes[1].plot(val_recon_losses, label="Val Recon Loss")
axes[1].set_title("CVAE Reconstruction Loss (MSE)")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

# KL Divergence
axes[2].plot(train_kl_losses, label="Train KL Loss")
axes[2].plot(val_kl_losses, label="Val KL Loss")
axes[2].set_title("CVAE KL Divergence")
axes[2].set_xlabel("Epochs")
axes[2].set_ylabel("KL")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Visualize original vs. reconstructed images on test set
model.eval()
with torch.no_grad():
    # Get one batch from test loader
    test_images, test_labels = next(iter(test_loader))
    test_images = test_images.to(device)
    test_labels = test_labels.to(device)
    reconstructed, _, _ = model(test_images, test_labels)

    num_display = min(8, test_images.size(0))
    fig, axes = plt.subplots(2, num_display, figsize=(2 * num_display, 4))

    for i in range(num_display):
        # Original
        orig_img = test_images[i].cpu().permute(1, 2, 0).numpy()
        orig_img = np.clip(orig_img, 0, 1)
        axes[0, i].imshow(orig_img)
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=10)

        # Reconstructed
        recon_img = reconstructed[i].cpu().permute(1, 2, 0).numpy()
        recon_img = np.clip(recon_img, 0, 1)
        axes[1, i].imshow(recon_img)
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=10)

    plt.suptitle("CVAE: Original vs Reconstructed (Test Set)", fontsize=14)
    plt.tight_layout()
    plt.show()


# 8. Conditional Image Generation

In [ ]:
# Generate images conditioned on specific class labels.
# This is the key advantage of CVAE over standard VAE:
# we can control which class of image to generate.
model.eval()

# Get class names for display
class_names = datasets.Food101(root=DATA_PATH, split="train").classes[:NUM_CLASSES]

num_classes_to_show = min(5, NUM_CLASSES)
num_samples_per_class = 4

fig, axes = plt.subplots(
    num_classes_to_show, num_samples_per_class,
    figsize=(3 * num_samples_per_class, 3 * num_classes_to_show),
)

with torch.no_grad():
    for row, class_idx in enumerate(range(num_classes_to_show)):
        # Sample z from standard normal distribution
        z = torch.randn(num_samples_per_class, LATENT_DIM).to(device)
        # Create label tensor for the target class
        labels = torch.full(
            (num_samples_per_class,), class_idx, dtype=torch.long
        ).to(device)

        # Decode conditioned on the class label
        generated = model.decode(z, labels)

        for col in range(num_samples_per_class):
            img = generated[col].cpu().permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(
                    class_names[class_idx], fontsize=10, rotation=0,
                    labelpad=80, ha="right",
                )

plt.suptitle("CVAE: Conditional Image Generation", fontsize=16)
plt.tight_layout()
plt.show()
